# 7 - Number of laps (nLaps)

In [11]:
%%capture
import os

# go to the project root (the notebook is in the "notebooks" folder)
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

# run the graphics part (which runs the reading part) to get x_c, y_c, r and theta
# %%capture hides its figures and prints (to remove when we merge the notebooks)
# get_ipython().run_line_magic("run", ...) is the same as %run, but works inside %%capture
if "theta_trials" not in globals():
    get_ipython().run_line_magic("run", "notebooks/02_graphics.ipynb")      

### Keep each record from the first entry into the target
As in MouseReMoCo, the statistics are computed only after the cursor enters
the target for the first time (first sample with mouseInTarget = 1).
We reuse the centered coordinates, r and θ computed in notebook 2 (02_graphics).

In [12]:
# data of each record, from the first entry into the target
t_kept, x_kept, y_kept, r_kept, theta_kept = [], [], [], [], []

for i in range(n_records):
    first = np.argmax(in_target_trials[i] == 1)   # index of the first sample inside the target

    t_kept.append(t_trials[i][first:])
    x_kept.append(x_c_trials[i][first:])       # centered x (from notebook 2)
    y_kept.append(y_c_trials[i][first:])       # centered y, pointing up (from notebook 2)
    r_kept.append(r_trials[i][first:])         # distance to the center (from notebook 2)
    theta_kept.append(theta_trials[i][first:]) # angle (from notebook 2)

    print(f"record {i}: first entry at {t_trials[i][first]:7.3f} s (sample {first}), "
          f"{len(t_kept[i])} samples kept")

record 0: first entry at   0.022 s (sample 0), 3971 samples kept
record 1: first entry at  40.966 s (sample 92), 3806 samples kept
record 2: first entry at  80.832 s (sample 58), 3840 samples kept
record 3: first entry at 120.908 s (sample 52), 3832 samples kept
record 4: first entry at 160.908 s (sample 93), 3843 samples kept


### Observations
- Record 0 starts directly inside the target (sample 0), so all its samples are kept.
- In records 1 to 4, the cursor enters the target after about 1 s (92, 58, 52 and 93 samples removed).
- About 3800 to 3970 samples are kept per record, so almost the whole 20 s of each record is used for the statistics.

### Number of laps
θ (computed in notebook 2 with arctan2) is the angle of the cursor around the center, between -π and π.
When the cursor crosses the left side of the circle, θ jumps from +π to -π
(e.g. 170°, 180°, -170° instead of 170°, 180°, 190°).
np.unwrap removes these jumps, so that θ keeps increasing (or decreasing) continuously.
The number of laps is then the total change of angle divided by one full turn (2π):
nLaps = (θ_end - θ_start) / 2π. A negative value means that θ decreases.

In [13]:
n_laps = []

for i in range(n_records):
    theta = np.unwrap(theta_kept[i])   # continuous angle (radians)
    n_laps.append((theta[-1] - theta[0]) / (2 * np.pi))  # total turn / one full turn

    print(f"record {i}: nLaps = {n_laps[i]:6.2f}")

record 0: nLaps = -10.84
record 1: nLaps = -11.97
record 2: nLaps = -13.26
record 3: nLaps = -12.83
record 4: nLaps = -14.74


### Results
We find the same values as in the "recomputed statistics" of the assignment (-10.84, -11.97, -13.26, -12.83, -14.74). They differ from the MouseReMoCo table by at most 0.01 lap (e.g. -10.84 vs -10.85), due to rounding.
The values are negative because we use a y axis pointing up (as in notebook 2):
θ decreases, so the cursor turns clockwise on the screen.
With a y axis pointing down (screen coordinates), the same movement would give positive values,
as in the former version of the assignment.

The number of laps increases overall over the session (from about 11 to 15 laps in 20 s),
but not at every record: record 3 (-12.83) has fewer laps than record 2 (-13.26).

# 8 - Compute Re, Te and error

Re stands for effective Radius which means the radius of the trajectory that the participant really follows. 

Te stands for effective Tolerance which means the effective width of the path around the circle (spread of r).

Error is the difference between the trajectory that the participant should follow and the trajectory that he really follows. More precisely, it is the percentage of the total angular displacement that occurs while the mouse is outside the target.

### Re

To compute Re, we simply compute the mean radius values for each record. This method is used in the textbook: https://perso.lisn.upsaclay.fr/chapuis/ihm-polytech/pointage.pdf, which explains Fitts' law. 

### Te

To compute Te, we adapted a formula found in https://perso.lisn.upsaclay.fr/chapuis/ihm-polytech/pointage.pdf. We calculate the standard deviation of the radius (r) and multiply by 4.133. We use (r) instead of (x), as in the original formula, because we want to measure the radial dispersion of the trajectory rather than the dispersion along a single axis. The value 4.133 is a statistical constant derived from the normal distribution. It is used to estimate an effective width that covers approximately 96% of the observed values, assuming a normal distribution.

### Error

To compute error, we divide the angular displacement outside the target by the total angular displacement, then multiply it to get a percentage.

In [105]:
Re, Te, error = [], [], []

for i in range(n_records):
    # Use the radius values already kept from the first entry into the target
    r = r_kept[i]

    # Unwrap the kept angles to obtain a continuous angular trajectory
    theta = np.unwrap(theta_kept[i])

    # Keep the target-membership data from the first entry into the target
    first = np.argmax(in_target_trials[i] == 1)
    in_target = in_target_trials[i][first:]

    # Calculate the absolute angular displacement between consecutive samples
    d_theta = np.abs(np.diff(theta))

    # Identify angular steps that end outside the target
    out = in_target[1:] == 0

    # Calculate the mean distance from the center (Re)
    Re.append(np.mean(r))

    # Calculate the variability of the distance from the center (Te)
    Te.append(4.133 * np.std(r))

    # Calculate the percentage of angular displacement outside the target
    error.append(100 * np.sum(d_theta[out]) / np.sum(d_theta))

    # Display the results for the current record
    print(
        f"record {i}: Re = {Re[i]:6.2f} px "
        f"Te = {Te[i]:6.2f} px "
        f"error = {error[i]:6.2f} %"
    )

record 0: Re = 213.86 px Te =  91.79 px error =   3.90 %
record 1: Re = 209.85 px Te =  34.45 px error =   0.74 %
record 2: Re = 209.43 px Te =  42.42 px error =   2.81 %
record 3: Re = 209.46 px Te =  37.14 px error =   0.13 %
record 4: Re = 210.89 px Te =  43.88 px error =   2.84 %


## Observations

By calculating the mean value of (r), we obtain exactly the same results as in the instructions for Re.

Regarding, Te we observed a really small difference for Te of record 0 (91.79 instead of 91.78) which is probably due to rounding. The rest of the values are exactly the same.

For the error, the values correspond exactly to the values provided in the instructions.